In [14]:
import pandas as pd

from common.utils import pipeline
from common.ETL import extract_df_from_db, save_mart

### Общие ETL шаги

In [15]:
df_deliveries = pipeline(
    table_name="deliveries",
    date_column="date",
)

df_drivers = pipeline(
    table_name="drivers", 
    transform_outliers=False
)

### Подготовка данных

In [16]:
df_logistics = df_deliveries.merge(
    df_drivers[["driver_id", "name", "experience_years", "region"]],
    on="driver_id",
    how="left",
)

df_logistics["route"] = df_logistics["source"] + " -> " + df_logistics["destination"]
df_logistics["cost_per_km"] = df_logistics["cost_usd"] / df_logistics["distance_km"]
df_logistics["is_delayed"] = df_logistics["delay_hours"] > 0

df_logistics.head(5)

,delivery_id,date,source,destination,product_type,volume_ton,cost_usd,delay_hours,distance_km,weather_conditions,driver_id,vehicle_id,name,experience_years,region,route,cost_per_km,is_delayed
0,1,2025-10-01,Base-Khanty,Station-01,Diesel,32.5,2100.50,0.0,180.0,Clear,1,1,Ivan Petrov,8,Khanty-Mansi,Base-Khanty -> Station-01,11.669444,False
1,2,2025-10-01,Base-Tomsk,Station-02,Gasoline,28.0,1850.00,1.5,150.0,Rain,2,2,Sergey Sidorov,12,Tomsk,Base-Tomsk -> Station-02,12.333333,True
2,3,2025-10-01,Base-Tyumen,Station-03,Diesel,22.0,1650.25,0.0,120.0,Clear,3,3,Aleksey Smirnov,5,Tyumen,Base-Tyumen -> Station-03,13.752083,False
3,4,2025-10-02,Base-Khanty,Station-04,Kerosene,35.0,2400.80,2.0,210.0,Fog,4,1,Pavel Egorov,15,Perm,Base-Khanty -> Station-04,11.432381,True
4,5,2025-10-02,Base-Tomsk,Station-05,Gasoline,20.5,1500.00,0.5,110.0,Cloudy,5,2,Andrey Kuznetsov,10,Omsk,Base-Tomsk -> Station-05,13.636364,True


### Факторы задержек: погода

In [17]:
mart_delay_weather = (
    df_logistics
    .groupby("weather_conditions", as_index=False)
    .agg(
        deliveries_count=("delivery_id", "count"),
        delayed_count=("is_delayed", "sum"),
        avg_delay_hours=("delay_hours", "mean"),
        max_delay_hours=("delay_hours", "max"),
    )
)

mart_delay_weather["delay_rate"] = mart_delay_weather["delayed_count"] / mart_delay_weather["deliveries_count"]
mart_delay_weather.sort_values("avg_delay_hours", ascending=False)

,weather_conditions,deliveries_count,delayed_count,avg_delay_hours,max_delay_hours,delay_rate
4,Snow,3,3,3.416667,3.75,1.000000
3,Rain,5,5,2.000000,3.00,1.000000
2,Fog,3,3,1.500000,2.00,1.000000
1,Cloudy,4,3,0.375000,0.50,0.750000
0,Clear,15,2,0.100000,1.00,0.133333


### Факторы задержек: расстояние

In [18]:
mart_delay_distance = df_logistics[[
    "delivery_id",
    "date",
    "route",
    "distance_km",
    "delay_hours",
    "weather_conditions",
    "driver_id",
    "name",
]].copy()

mart_delay_distance.head(5)

,delivery_id,date,route,distance_km,delay_hours,weather_conditions,driver_id,name
0,1,2025-10-01,Base-Khanty -> Station-01,180.0,0.0,Clear,1,Ivan Petrov
1,2,2025-10-01,Base-Tomsk -> Station-02,150.0,1.5,Rain,2,Sergey Sidorov
2,3,2025-10-01,Base-Tyumen -> Station-03,120.0,0.0,Clear,3,Aleksey Smirnov
3,4,2025-10-02,Base-Khanty -> Station-04,210.0,2.0,Fog,4,Pavel Egorov
4,5,2025-10-02,Base-Tomsk -> Station-05,110.0,0.5,Cloudy,5,Andrey Kuznetsov


### Анализ стоимости: cost / km

In [19]:
mart_cost_distance = df_logistics[[
    "delivery_id",
    "date",
    "route",
    "distance_km",
    "cost_usd",
    "cost_per_km",
    "volume_ton",
    "product_type",
]].copy()

mart_cost_distance.head(5)

,delivery_id,date,route,distance_km,cost_usd,cost_per_km,volume_ton,product_type
0,1,2025-10-01,Base-Khanty -> Station-01,180.0,2100.50,11.669444,32.5,Diesel
1,2,2025-10-01,Base-Tomsk -> Station-02,150.0,1850.00,12.333333,28.0,Gasoline
2,3,2025-10-01,Base-Tyumen -> Station-03,120.0,1650.25,13.752083,22.0,Diesel
3,4,2025-10-02,Base-Khanty -> Station-04,210.0,2400.80,11.432381,35.0,Kerosene
4,5,2025-10-02,Base-Tomsk -> Station-05,110.0,1500.00,13.636364,20.5,Gasoline


### Факторы задержек: водитель / KPI по водителям

In [20]:
mart_driver_kpi = (
    df_logistics
    .groupby(["driver_id", "name", "experience_years", "region"], as_index=False)
    .agg(
        deliveries_count=("delivery_id", "count"),
        avg_delay_hours=("delay_hours", "mean"),
        delayed_count=("is_delayed", "sum"),
        avg_cost_per_km=("cost_per_km", "mean"),
        total_volume_ton=("volume_ton", "sum"),
        total_distance_km=("distance_km", "sum"),
    )
)

mart_driver_kpi["delay_rate"] = mart_driver_kpi["delayed_count"] / mart_driver_kpi["deliveries_count"]
mart_driver_kpi.sort_values(["avg_delay_hours", "avg_cost_per_km"])

,driver_id,name,experience_years,region,deliveries_count,avg_delay_hours,delayed_count,avg_cost_per_km,total_volume_ton,total_distance_km,delay_rate
0,1,Ivan Petrov,8,Khanty-Mansi,5,0.200000,1,11.726857,168.4,941.0,0.200000
2,3,Aleksey Smirnov,5,Tyumen,7,0.428571,1,13.261150,165.1,895.0,0.142857
3,4,Pavel Egorov,15,Perm,4,0.625000,2,11.754716,130.7,737.0,0.500000
1,2,Sergey Sidorov,12,Tomsk,7,0.785714,5,12.299379,195.2,1053.0,0.714286
4,5,Andrey Kuznetsov,10,Omsk,7,2.250000,7,13.587001,141.1,766.0,1.000000


### Оптимизация маршрутов

In [21]:
mart_route_optimization = (
    df_logistics
    .groupby(["source", "destination", "route"], as_index=False)
    .agg(
        deliveries_count=("delivery_id", "count"),
        avg_delay_hours=("delay_hours", "mean"),
        avg_cost_per_km=("cost_per_km", "mean"),
        avg_distance_km=("distance_km", "mean"),
        total_volume_ton=("volume_ton", "sum"),
    )
)

mart_route_optimization["optimization_score"] = mart_route_optimization["avg_delay_hours"] + mart_route_optimization["avg_cost_per_km"]

mart_route_optimization.sort_values("optimization_score")

,source,destination,route,deliveries_count,avg_delay_hours,avg_cost_per_km,avg_distance_km,total_volume_ton,optimization_score
8,Base-Khanty,Station-27,Base-Khanty -> Station-27,1,0.00,11.500000,200.0,35.0,11.500000
0,Base-Khanty,Station-01,Base-Khanty -> Station-01,1,0.00,11.669444,180.0,32.5,11.669444
6,Base-Khanty,Station-19,Base-Khanty -> Station-19,1,0.00,11.722581,186.0,33.5,11.722581
3,Base-Khanty,Station-09,Base-Khanty -> Station-09,1,0.00,11.770000,170.0,31.0,11.770000
5,Base-Khanty,Station-15,Base-Khanty -> Station-15,1,0.00,11.816484,182.0,32.9,11.816484
2,Base-Khanty,Station-06,Base-Khanty -> Station-06,1,0.00,11.895946,185.0,33.2,11.895946
18,Base-Tomsk,Station-12,Base-Tomsk -> Station-12,1,0.00,11.935484,155.0,26.8,11.935484
21,Base-Tomsk,Station-24,Base-Tomsk -> Station-24,1,0.00,12.080537,149.0,26.5,12.080537
19,Base-Tomsk,Station-16,Base-Tomsk -> Station-16,1,0.50,11.866667,150.0,27.0,12.366667
7,Base-Khanty,Station-23,Base-Khanty -> Station-23,1,0.50,12.000000,175.0,31.8,12.500000


### Сохранение витрин в БД

In [22]:
save_mart(mart_delay_weather, "mart_delay_weather")
save_mart(mart_cost_distance, "mart_cost_distance")
save_mart(mart_driver_kpi, "mart_driver_kpi")